# Procesando data para importar a MATLAB

In [2]:
##### Importacion de modulos
import pandas as pd
import numpy as np
from tqdm import tqdm
import scipy

In [3]:
##### Etiquetando retornos logaritmicos
#### Cargando datos
df_rendis_log = pd.read_csv('retornos_log_splac.csv', index_col='Date')

#### Creando una columna de la mediana diaria
df_rendis_log['Median'] = df_rendis_log.median(axis=1)

#### Creando etiquetas 
for nombre in df_rendis_log.columns.to_list()[:-1]:
    comparacion = df_rendis_log[nombre] > df_rendis_log['Median']
    df_rendis_log['Label.' + nombre] = comparacion.astype(int)
df_rendis_log = df_rendis_log.drop('Median', axis=1)
    # Eliminar pq no es necesaria despues


df_rendis_log.head()
df_rendis_log.info()

<class 'pandas.DataFrame'>
Index: 5291 entries, 2006-02-28 to 2026-06-12
Data columns (total 64 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   ABEV             5291 non-null   float64
 1   AC               5291 non-null   float64
 2   AMXB             5291 non-null   float64
 3   AXIA3            5291 non-null   float64
 4   BAP              5291 non-null   float64
 5   BBAS3            5291 non-null   float64
 6   BBD              5291 non-null   float64
 7   BIMBOA           5291 non-null   float64
 8   BSAC             5291 non-null   float64
 9   CEMEXCPO         5291 non-null   float64
 10  CENCOSUD         5291 non-null   float64
 11  CIB              5291 non-null   float64
 12  FALABELLA        5291 non-null   float64
 13  FEMSAUBD         5291 non-null   float64
 14  GCARSOA1         5291 non-null   float64
 15  GFNORTEO         5291 non-null   float64
 16  GGB              5291 non-null   float64
 17  GMEXICOB       

In [4]:
##### Creando secuencias de datos y separando los ingresos y etiquetas de datos
def matlab_procesar(df:pd.DataFrame, tamano_ventana: int = 30):
    ventanas_ingresantes = []; etiquetas_de_ventanas = []
        # 1 ventana tendra 'tamano_ventana' dias de datos
        # 1 etiqueta por ventana tendra 1 solo dia de dato
        # I.e., el modelo procesara 30 dias para predicir el 31 dia
    num_registros, num_variables = df.shape
    mitad = int(num_variables/2)
        # Mitad de los datos son ingresos y la otra mitad son etiquetas
    
    for i in tqdm(range(num_registros - tamano_ventana), desc='matlabProcesar', position=1, leave=False):
        df_ventana = df.iloc[i: i+tamano_ventana, :mitad]
            # Tiene 30 filas, representando 30 dias
            # Consiste solamente de los retornos logaritmicos
            # Los ingresos son los precios de cada empresa diaria
        df_etiqueta = df.iloc[i+tamano_ventana, mitad:]
            # Tiene 1 fila, representando 1 dia
            # Consiste solamente de los valores binarios
                # Representando si la empresa tiene retornos mayores al retorno mediano
            
        ventanas_ingresantes.append(df_ventana.to_numpy())
        etiquetas_de_ventanas.append(df_etiqueta.to_numpy())
    
    return np.array(ventanas_ingresantes), np.array(etiquetas_de_ventanas)


In [ ]:
##### Tenica de evaluacion: sliding window split
def sliding_window_split(
        df: pd.DataFrame, 
        train_anos = 1, 
        val_anos = 0.25, 
        test_anos = 1, 
        input_window_size = 30
    ):
    """
    Separa los datos usando la tecnica de sliding window

    :param df: Retornos logaritmicos normalizados
    :param train_anos: tamano de ventana para entrenamiento
    :param test anos: tamano de ventana para prueba
    :return: Lista de (trainX, trainY, valX, valY, testX, testY) tuples
    """
    dias_por_ano = 252
        # Aproximadamente numero de dias para hacer trading
    dias_train = int(dias_por_ano * train_anos)
    dias_val = int(dias_por_ano * val_anos)
    dias_test = int(dias_por_ano * test_anos)

    num_registros = len(df)
    acumulacion = []
    prev = 0

    for i in tqdm(
        range(0, num_registros, dias_train - input_window_size),
        desc = 'slidingSplit',
        position = 0
    ):
        train_empezar = prev
        train_terminar = train_empezar + dias_train
        val_empezar = train_terminar
        val_terminar = val_empezar + dias_val
        test_empezar = val_terminar
        test_terminar = test_empezar + dias_test

        if test_terminar > num_registros:
            # No se quiere tests que no sean el mismo tamano
            print(f'\n\nIteracion {i} - dias faltantes en ultimo set: {test_terminar - num_registros}')
            print(f'\nLos ultimos {num_registros - test_empezar - 30} se ignoraran')
                #"-30" pq la ventana fue considerada en la ultima iteracion

        train_data = df.iloc[train_empezar:train_terminar]
        val_data = df.iloc[val_empezar:val_terminar]
        test_data = df.iloc[test_empezar:test_terminar]
        train_ventanas, train_etiquetas = matlab_procesar(train_data, input_window_size)
        val_ventanas, val_etiquetas = matlab_procesar(val_data, input_window_size)
        test_ventanas, test_etiquetas = matlab_procesar(test_data, input_window_size)
        acumulacion.append((train_ventanas, train_etiquetas, val_ventanas, val_etiquetas, test_ventanas, test_etiquetas))

        prev = train_terminar - input_window_size
            #Toma en consideracion el tamano de la ventana
    return acumulacion


In [6]:
##### Transformando sliding split en .mat
def sliding_to_mat(ds_lista: list[tuple], fname: str):
    """ 
    Convierte sliding hacia .mat archivo
    """
    if not isinstance(ds_lista, list):
        raise TypeError("df_list debe ser una lista de tuples.")
    
    ds_dict = {}
    for i, tup in enumerate(ds_lista):
        if not isinstance(tup, tuple):
            raise TypeError("tup debe ser tuple de train,val,test")
        set_data = {}
        for j, train_val_test in enumerate(tup):
            set_data[f'set_{j}'] = train_val_test
        ds_dict[f'ds_{i}'] = set_data

    scipy.io.savemat(fname, ds_dict)
    print('termino')
    
    

In [ ]:
##### Tecnica de evaluacion: expanding window split
def expanding_window_split(
        df: pd.DataFrame,
        train_anos = 1,
        val_anos = 0.25,
        test_anos = 1, 
        input_window_size = 30
):
    """
    Separa data usando expanding window split
    df: retornos logaritmicos
    train_anos: rango de anos que se va agregar en cada expansion
    val_anos: anos en validation set
    test_anos : cantidad de anos en el test set
    :return: Lista de (trainX, trainY, valX, valY, testX, testY) tuples
    """
    dias_por_ano = 252
    dias_train = int(dias_por_ano * train_anos)
    dias_val = int(dias_por_ano * val_anos)
    dias_test = int(dias_por_ano * test_anos)

    num_registros = len(df)
    acumulacion = []

    prev_train_terminar = 0

    for i in tqdm(
        range(0, num_registros, dias_train - input_window_size),
        desc = 'slidingSplit',
        position = 0
    ):
        train_empezar = 0  #Expanding siempre empieza de 0
        train_terminar = prev_train_terminar + dias_train
        val_empezar = train_terminar
        val_terminar = val_empezar + dias_val
        test_empezar = val_terminar
        test_terminar = test_empezar + dias_test

        if test_terminar > num_registros:
            # No se quiere tests que no sean el mismo tamano
            print(f'\n\nIteracion {i} - dias faltantes en ultimo set: {test_terminar - num_registros}')
            print(f'\nLos ultimos {num_registros - test_empezar - 30} se ignoraran')
                #"-30" pq la ventana fue considerada en la ultima iteracion
            break

        train_data = df.iloc[train_empezar:train_terminar]
        val_data = df.iloc[val_empezar:val_terminar]
        test_data = df.iloc[test_empezar:test_terminar]
        train_ventanas, train_etiquetas = matlab_procesar(train_data, input_window_size)
        val_ventanas, val_etiquetas = matlab_procesar(val_data, input_window_size)
        test_ventanas, test_etiquetas = matlab_procesar(test_data, input_window_size)
        acumulacion.append((train_ventanas, train_etiquetas, val_ventanas, val_etiquetas, test_ventanas, test_etiquetas))

        prev_train_terminar = train_terminar - input_window_size

    return acumulacion



In [8]:
##### Covnertiendo de expanding a mat
def expanding_to_mat(ds_lista: list[tuple], fname:str, parte:int):
    """
    Convierte Expansing a .mat
    Se necesita 2 partes por el tamano del archivo
    """

    if not isinstance(ds_lista, list):
        raise TypeError("df_list debe ser una lista de tuples.")
    
    ds_dict = {}
    parte_break = 15
    for i, tup in enumerate(ds_lista):
        if parte == 1:
            if i == parte_break: break
        else: 
            if i < parte_break: continue
        if not isinstance(tup, tuple):
            raise TypeError("tup debe ser tuple de train,val,test")
        set_data = {}
        for j, train_val_test in enumerate(tup):
            set_data[f'set_{j}'] = train_val_test
        ds_dict[f'ds_{i}'] = set_data

    scipy.io.savemat(fname, ds_dict)
    print('termino')

In [11]:
##### Cumulacion: df_rendis_log a matlab - Sliding tecnica
sliding_arr = sliding_window_split(df_rendis_log)
# sliding_to_mat(sliding_arr, 'sliding_v2.mat')

slidingSplit:  92%|█████████▏| 22/24 [00:09<00:00,  2.38it/s]



Iteracion 4884 - dias faltantes en ultimo set: 160

Los ultimos 92 se ignoraran


In [10]:
##### Cumulacion: df_rendis_log a matlab - expanding tecnica
# expanding_arr = expanding_window_split(df_rendis_log)
# expanding_to_mat(expanding_arr,'expanding_1_v2.mat',1)
# expanding_to_mat(expanding_arr,'expanding_2_v2.mat',2)